# 🧠 Pillar B: Yuan-Yuan's AI Knowledge Base Assistant
**Project:** Yuan-Yuan's AI Knowledge Base  
**System Role:** Retrieval-Augmented Generation (RAG) Interface & Evaluator

### 🎯 Objective
This is the user-facing interface. It connects the persistent ChromaDB vector store to Azure OpenAI to provide accurate, cited answers based strictly on Yuan-Yuan's curated knowledge base.

### 🛠️ Key Architectural Features:
1. **Semantic Retrieval:** Uses vector similarity to find relevant context, allowing the assistant to understand the "intent" behind a question rather than just keywords.
2. **Hallucination Guardrails:** Implements strict system prompts that require the AI to cite sources and admit ignorance if the information is missing from the context.
3. **Low-Latency Architecture:** Decouples heavy data processing (Pillar A) from the user interface for near-instant response times.
4. **BERT Semantic Evaluation:** Uses a local transformer model to calculate the similarity between the question and the response, providing a quantitative "Confidence Score."

---
*Status: Ready for Querying. Operates independently of the Confluence API.*

In [16]:
# --- STEP 1: ENVIRONMENT & UTILITIES ---
# This cell must handle the "Environment Handshake" and ensure SQLite is upgraded and rag_util is loaded correctly. 
import pysqlite3
import sys
import os

# Essential SQLite override for ChromaDB compatibility
sys.modules["sqlite3"] = pysqlite3

import chromadb
from openai import AzureOpenAI
import rag_util  # This loads your model and cleaning logic

from sentence_transformers import util  # For answer quality verification
# Initialize Azure Client
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

print("✅ Environment Ready: SQLite Overridden, Azure Connected, rag_util Loaded.")

✅ Environment Ready: SQLite Overridden, Azure Connected, rag_util Loaded.


In [22]:
# --- SYSTEM HEALTH CHECK ---
def run_health_check():
    try:
        # Connect to the persistent store
        db_client = chromadb.PersistentClient(path="./chroma_data")
        collection = db_client.get_collection(name="document_store")
        
        # 1. Count Total Chunks
        total_chunks = collection.count()
        
        # 2. Count Unique Pages (extracted from metadata)
        all_data = collection.get(include=['metadatas'])
        unique_pages = set(m['source_title'] for m in all_data['metadatas'])
        total_pages = len(unique_pages)
        
        print("📊 SYSTEM STATUS: ONLINE")
        print("-" * 25)
        print(f"✅ Knowledge Base: Linked (./chroma_data)")
        print(f"📑 Curated Pages:  {total_pages}")
        print(f"🧩 Searchable Chunks: {total_chunks}")
        print("-" * 25)
        
    except Exception as e:
        print(f"⚠️ Health Check Failed: {e}")

run_health_check()

📊 SYSTEM STATUS: ONLINE
-------------------------
✅ Knowledge Base: Linked (./chroma_data)
📑 Curated Pages:  9
🧩 Searchable Chunks: 56
-------------------------


In [ ]:
# --- STEP 2: THE ENGINE FUNCTIONS ---
# These functions are workhorse functions. They handle retrieval, prompt formatting, and LLM interaction.
def get_context_for_llm(question, n_results=3):
    """Retrieves 3 relevant chunks from your 56-chunk collection."""
    db_client = chromadb.PersistentClient(path="./chroma_data")
    collection = db_client.get_collection(name="document_store")
    
    # Use the model embedded in rag_util
    query_vector = rag_util.model.encode([question])[0].tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=n_results)
    
    context_output = ""
    for i in range(len(results['documents'][0])):
        source = results['metadatas'][0][i]['source_title']
        content = results['documents'][0][i]
        context_output += f"\n--- FROM DOCUMENT: {source} ---\n{content}\n"
    return context_output

def generate_migration_advice(question, context):
    """Formats the prompt for Yuan-Yuan's Assistant."""
    return f"""
    You are a Yuan-Yuan's confluence resource AI assistant. 
    Use the following retrieved context to answer the user's question.
    
    RULES:
    1. If the answer is not in the context, say "I don't have documentation on that."
    2. Use a professional, technical tone.
    3. Always cite the 'SOURCE' provided in the context.

    CONTEXT:
    {context}

    USER QUESTION: {question}
    
    EXPERT MIGRATION ADVICE:
    """

def ask_yuan_yuan(question):
    """The master function that runs the whole chain."""
    context = get_context_for_llm(question)
    prompt = generate_migration_advice(question, context)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

def get_answer_confidence(question, ai_answer):
    """
    BERT Semantic Evaluation:
    Calculates how well the answer matches the user's intent.
    """
    # 1. Convert question and answer into vectors using your local model
    q_vector = rag_util.model.encode(question, convert_to_tensor=True)
    a_vector = rag_util.model.encode(ai_answer, convert_to_tensor=True)
    
    # 2. Compute Cosine Similarity (Math-based semantic match)
    cosine_scores = util.cos_sim(q_vector, a_vector)
    score = cosine_scores.item()
    
    # 3. Assign a label
    if score > 0.75:
        label = "✅ High Confidence"
    elif score > 0.50:
        label = "⚠️ Moderate Confidence"
    else:
        label = "❌ Low Confidence / Verify Source"
        
    return score, label
# Evaluation. This function uses the local BERT model to check if the AI's answer semantically matches the user's intent, 
# providing a similarity score and a confidence label.
def ask_yuan_yuan_safe(question):
    try:
        # Check if environment variables are actually loaded
        if not os.getenv("AZURE_OPENAI_KEY"):
            return "❌ Error: API Key not found. Please check your environment variables."

        # 1. Retrieve
        context = get_context_for_llm(question)
        if not context.strip():
            return "🔍 I searched the knowledge base but couldn't find relevant documentation."

        # 2. Generate
        prompt = generate_migration_advice(question, context)
        
        # 3. Call Azure with a timeout
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        
        ai_answer = response.choices[0].message.content
        
        # 4. Evaluate
        score, label = get_answer_confidence(question, ai_answer)
        
        return f"{ai_answer}\n\n📊 EVALUATION: {score:.2f} ({label})"

    except Exception as e:
        return f"⚠️ System Note: I encountered an issue connecting to the AI brain. Error: {str(e)}"

In [20]:
# --- STEP 3: INTERFACE ---
# This is where you can ask questions and get answers. The function `ask_yuan_yuan` will handle the entire retrieval and response generation process.
user_query = "What VS Code extensions do I need for SAS?"

answer = ask_yuan_yuan_safe(user_query)
print(answer)

To effectively edit SAS code in Visual Studio Code, you will need to install the SAS extension published by SAS Institute Inc. This extension provides functionality similar to the traditional PC SAS code editor. 

Here are the setup steps for installing the necessary extension:

1. Download the Visual Studio Code User Installer from the Microsoft website.
2. Install Visual Studio Code.
3. Open Visual Studio Code.
4. On the left panel, select Extensions.
5. Search for the SAS extension published by SAS Institute Inc. and install it.
6. If you wish for the editor to resemble the traditional SAS editor, set the color theme to SAS Light.

Please note that if you encounter an error indicating that the extension is not compatible with your current version of VS Code, you will need to upgrade Visual Studio Code to the latest version. Uninstall the existing version, download the newest version from the Microsoft website, and install it again.

For further details, refer to the document "Work w